# Solutions: Notebook 08 — Monotonicity and Non-Crossing Quantiles

Complete solutions with explanations for all six exercises.

In [ ]:
# Standard libraries
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Statistical libraries
from scipy import stats

# PanelBox imports
from panelbox.models.quantile import PooledQuantile
from panelbox.models.quantile.monotonicity import (
    QuantileMonotonicity,
)

# Visualization configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

# Reproducibility
np.random.seed(42)

# Define paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
RESULTS_DIR = OUTPUT_DIR / "results"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
# Helper: SimpleResult class for monotonicity functions
class SimpleResult:
    """Lightweight result object compatible with QuantileMonotonicity."""

    def __init__(self, params, X):
        self.params = params
        self.model = type("Model", (), {"X": X, "nobs": len(X)})()


# Load crossing example data for later exercises
data = pd.read_csv(DATA_DIR / "simulated" / "crossing_example.csv")
y = data["y"].values
X = np.column_stack([np.ones(len(data)), data["x1"].values, data["x2"].values])
entity_id = data["id"].values
var_names = ["const", "x1", "x2"]

# Estimate unconstrained QR
tau_grid = np.round(np.arange(0.1, 0.91, 0.1), 2)
qr_results = {}
results_simple = {}
for tau in tau_grid:
    model = PooledQuantile(y, X, entity_id=entity_id, quantiles=tau)
    qr_results[tau] = model.fit(se_type="cluster")
    results_simple[tau] = SimpleResult(qr_results[tau].params.ravel(), X)

print(f"Loaded data: {data.shape}")
print(f"Estimated {len(tau_grid)} quantile models.")

---

## Exercise 1: Manual Rearrangement (Easy)

**Task**: Implement the rearrangement method manually without using `QuantileMonotonicity.rearrangement()`.

In [ ]:
# SOLUTION: Manual rearrangement

# Step 1: Get predictions for all observations and all quantiles
n_obs = len(X)
n_tau = len(tau_grid)

# predictions shape: (n_obs, n_tau)
predictions = np.column_stack([X @ results_simple[tau].params for tau in tau_grid])

print(f"Predictions matrix shape: {predictions.shape}  (n_obs x n_tau)")

# Check for crossings before rearrangement
n_crossing_before = 0
for i in range(n_obs):
    for j in range(n_tau - 1):
        if predictions[i, j] > predictions[i, j + 1] + 1e-10:
            n_crossing_before += 1

print(f"Number of crossing violations before rearrangement: {n_crossing_before}")

# Step 2: For each observation, sort the predictions
rearranged_predictions = np.zeros_like(predictions)
for i in range(n_obs):
    rearranged_predictions[i] = np.sort(predictions[i])

# Step 3: Verify monotonicity
n_crossing_after = 0
for i in range(n_obs):
    for j in range(n_tau - 1):
        if rearranged_predictions[i, j] > rearranged_predictions[i, j + 1] + 1e-10:
            n_crossing_after += 1

print(f"Number of crossing violations after rearrangement: {n_crossing_after}")

# Step 4: Compare with PanelBox implementation
pb_rearranged = QuantileMonotonicity.rearrangement(results_simple, X)

# Get PanelBox rearranged predictions
pb_preds = np.column_stack([X @ pb_rearranged[tau].params for tau in tau_grid])

# Note: PanelBox uses lstsq to find coefficients that approximate the sorted predictions,
# so the results may differ slightly at the coefficient level but be close at prediction level.
max_diff = np.max(np.abs(rearranged_predictions - pb_preds))
mean_diff = np.mean(np.abs(rearranged_predictions - pb_preds))
print("\nComparison with PanelBox:")
print(f"  Max absolute difference in predictions: {max_diff:.6f}")
print(f"  Mean absolute difference in predictions: {mean_diff:.6f}")
print("  Note: Small differences expected because PanelBox solves lstsq")
print("  to find coefficients that approximate sorted predictions.")

# Show a few examples
print("\nExample (observation 0):")
print(f"  Original:       {predictions[0]}")
print(f"  Manual sorted:  {rearranged_predictions[0]}")
print(f"  PanelBox:       {pb_preds[0]}")

---

## Exercise 2: Crossing from Small Samples vs Misspecification (Easy)

**Task**: Show that rearrangement fixes crossing from finite samples but not from model misspecification.

In [ ]:
# SOLUTION: Two scenarios
np.random.seed(42)

tau_test = np.round(np.arange(0.1, 0.91, 0.1), 2)

# ===== Scenario (a): Small sample, correct model =====
n_small = 30
x_small = np.random.normal(0, 1, n_small)
# Generate y with heteroskedastic errors to encourage crossing
eps_small = np.random.normal(0, 1, n_small) * (1 + 0.5 * np.abs(x_small))
y_small = 2 + 3 * x_small + eps_small
X_small = np.column_stack([np.ones(n_small), x_small])

# Estimate QR
results_small = {}
for tau in tau_test:
    model = PooledQuantile(y_small, X_small, quantiles=tau)
    results_small[tau] = SimpleResult(model.fit(se_type="nonrobust").params.ravel(), X_small)

# Detect crossing
report_small = QuantileMonotonicity.detect_crossing(results_small, X_small)
print("SCENARIO (a): Small sample, correct model")
print(f"  Has crossing: {report_small.has_crossing}")
print(f"  Pct affected: {report_small.pct_affected:.1f}%")

# Apply rearrangement
rearranged_small = QuantileMonotonicity.rearrangement(results_small, X_small)
report_small_rearr = QuantileMonotonicity.detect_crossing(rearranged_small, X_small)
print(f"  After rearrangement — crossing: {report_small_rearr.has_crossing}")

print()

In [ ]:
# ===== Scenario (b): Large sample, misspecified model =====
n_large = 2000
x_large = np.random.uniform(-3, 3, n_large)
# True model is quadratic with heteroskedastic errors
eps_large = np.random.normal(0, 1, n_large) * (0.5 + 0.8 * x_large**2)
y_large = 1 + 2 * x_large - 0.5 * x_large**2 + eps_large

# Fit LINEAR model (misspecified — omitting x^2)
X_large_mispec = np.column_stack([np.ones(n_large), x_large])

results_mispec = {}
for tau in tau_test:
    model = PooledQuantile(y_large, X_large_mispec, quantiles=tau)
    results_mispec[tau] = SimpleResult(
        model.fit(se_type="nonrobust").params.ravel(), X_large_mispec
    )

report_mispec = QuantileMonotonicity.detect_crossing(results_mispec, X_large_mispec)
print("SCENARIO (b): Large sample, misspecified model")
print(f"  Has crossing: {report_mispec.has_crossing}")
print(f"  Pct affected: {report_mispec.pct_affected:.1f}%")

# Apply rearrangement
rearranged_mispec = QuantileMonotonicity.rearrangement(results_mispec, X_large_mispec)
report_mispec_rearr = QuantileMonotonicity.detect_crossing(rearranged_mispec, X_large_mispec)
print(f"  After rearrangement — crossing: {report_mispec_rearr.has_crossing}")
print(f"  After rearrangement — pct affected: {report_mispec_rearr.pct_affected:.1f}%")

In [ ]:
# Visualize both scenarios
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Small sample
for obs_idx in range(min(10, n_small)):
    preds_before = [
        float(X_small[obs_idx : obs_idx + 1] @ results_small[tau].params) for tau in tau_test
    ]
    preds_after = [
        float(X_small[obs_idx : obs_idx + 1] @ rearranged_small[tau].params) for tau in tau_test
    ]
    axes[0].plot(tau_test, preds_before, "r-", alpha=0.3, linewidth=1)
    axes[0].plot(tau_test, preds_after, "b-", alpha=0.3, linewidth=1)

axes[0].plot([], [], "r-", linewidth=2, label="Before rearrangement")
axes[0].plot([], [], "b-", linewidth=2, label="After rearrangement")
axes[0].set_xlabel("Quantile (\u03c4)", fontsize=12)
axes[0].set_ylabel("Predicted Value", fontsize=12)
axes[0].set_title("(a) Small Sample \u2014 Rearrangement Fixes It", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Panel B: Misspecified model
sample_idx_b = np.random.choice(n_large, 15, replace=False)
for obs_idx in sample_idx_b:
    preds_before = [
        float(X_large_mispec[obs_idx : obs_idx + 1] @ results_mispec[tau].params)
        for tau in tau_test
    ]
    preds_after = [
        float(X_large_mispec[obs_idx : obs_idx + 1] @ rearranged_mispec[tau].params)
        for tau in tau_test
    ]
    axes[1].plot(tau_test, preds_before, "r-", alpha=0.3, linewidth=1)
    axes[1].plot(tau_test, preds_after, "b-", alpha=0.3, linewidth=1)

axes[1].plot([], [], "r-", linewidth=2, label="Before rearrangement")
axes[1].plot([], [], "b-", linewidth=2, label="After rearrangement")
axes[1].set_xlabel("Quantile (\u03c4)", fontsize=12)
axes[1].set_ylabel("Predicted Value", fontsize=12)
axes[1].set_title(
    "(b) Misspecification \u2014 Rearrangement Doesn't Fix Root Cause",
    fontsize=13,
    fontweight="bold",
)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("  (a) Small sample crossing is due to noise \u2014 rearrangement eliminates it.")
print("  (b) Misspecification crossing reflects a fundamental modeling problem.")
print("      Rearrangement can fix monotonicity at the prediction level but")
print("      the underlying coefficient inconsistency persists.")
print("      The correct fix is to improve the model (e.g., add x^2 term).")

---

## Exercise 3: Computational Time Comparison (Medium)

**Task**: Compare computational time of rearrangement, isotonic regression, and constrained QR.

In [ ]:
# SOLUTION: Computational time comparison
np.random.seed(42)

# Use the crossing example data
timing_results = {}

# 1. Time unconstrained QR estimation
t0 = time.time()
unconstrained_res = {}
for tau in tau_grid:
    model = PooledQuantile(y, X, entity_id=entity_id, quantiles=tau)
    unconstrained_res[tau] = SimpleResult(model.fit(se_type="cluster").params.ravel(), X)
t_unconstrained = time.time() - t0
timing_results["Unconstrained QR"] = t_unconstrained
print(f"Unconstrained QR: {t_unconstrained:.4f}s")

# 2. Time rearrangement (post-estimation correction only)
t0 = time.time()
_ = QuantileMonotonicity.rearrangement(unconstrained_res, X)
t_rearrangement = time.time() - t0
timing_results["Rearrangement (correction only)"] = t_rearrangement
timing_results["Rearrangement (total)"] = t_unconstrained + t_rearrangement
print(f"Rearrangement correction: {t_rearrangement:.4f}s")
print(f"Rearrangement total: {t_unconstrained + t_rearrangement:.4f}s")

# 3. Time isotonic regression (post-estimation correction only)
coef_mat = np.array([unconstrained_res[tau].params for tau in tau_grid])
t0 = time.time()
_ = QuantileMonotonicity.isotonic_regression(coef_mat, tau_grid)
t_isotonic = time.time() - t0
timing_results["Isotonic (correction only)"] = t_isotonic
timing_results["Isotonic (total)"] = t_unconstrained + t_isotonic
print(f"Isotonic correction: {t_isotonic:.4f}s")
print(f"Isotonic total: {t_unconstrained + t_isotonic:.4f}s")

# 4. Time constrained QR (use subsample for speed — full dataset is too slow)
np.random.seed(42)
n_sub = min(100, len(y))
sub_idx = np.random.choice(len(y), n_sub, replace=False)
X_sub = X[sub_idx]
y_sub = y[sub_idx]

t0 = time.time()
try:
    _ = QuantileMonotonicity.constrained_qr(X_sub, y_sub, tau_grid, method="SLSQP", max_iter=200)
    t_constrained = time.time() - t0
    timing_results["Constrained QR"] = t_constrained
    print(f"Constrained QR (n={n_sub} subsample): {t_constrained:.4f}s")
except Exception as e:
    t_constrained = None
    print(f"Constrained QR failed: {e}")

# 5. Time simultaneous QR (also use subsample)
t0 = time.time()
try:
    _ = QuantileMonotonicity.simultaneous_qr(X_sub, y_sub, tau_grid, lambda_nc=5.0, max_iter=200)
    t_simult = time.time() - t0
    timing_results["Simultaneous QR"] = t_simult
    print(f"Simultaneous QR (n={n_sub} subsample): {t_simult:.4f}s")
except Exception as e:
    t_simult = None
    print(f"Simultaneous QR failed: {e}")

In [ ]:
# Create bar chart comparing total times
total_methods = {
    "Unconstrained": t_unconstrained,
    "Rearrangement\n(total)": t_unconstrained + t_rearrangement,
    "Isotonic\n(total)": t_unconstrained + t_isotonic,
}
if t_constrained is not None:
    total_methods["Constrained\nQR"] = t_constrained
if t_simult is not None:
    total_methods["Simultaneous\nQR"] = t_simult

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(
    total_methods.keys(),
    total_methods.values(),
    color=["steelblue", "#009E73", "#E69F00", "#D55E00", "#CC79A7"][: len(total_methods)],
    edgecolor="black",
    alpha=0.8,
)

# Add time labels on bars
for bar, val in zip(bars, total_methods.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.3f}s",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

ax.set_ylabel("Time (seconds)", fontsize=12, fontweight="bold")
ax.set_title("Computational Time Comparison", fontsize=14, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print(f"  - Rearrangement correction time: {t_rearrangement:.4f}s (negligible overhead)")
print(f"  - Isotonic correction time: {t_isotonic:.4f}s (negligible overhead)")
if t_constrained:
    ratio = t_constrained / (t_unconstrained + t_rearrangement)
    print(f"  - Constrained QR is {ratio:.1f}x slower than unconstrained + rearrangement")
print("  - Recommendation: Use rearrangement for speed, constrained only when needed.")

---

## Exercise 4: Application to Real Data (Medium)

**Task**: Apply all three methods to the Card education data.

In [ ]:
# SOLUTION: Apply methods to Card education data
np.random.seed(42)

# Load data
card_data = pd.read_csv(DATA_DIR / "card_education.csv")
print(f"Card education data: {card_data.shape}")

# Prepare design matrix
y_card = card_data["lwage"].values
X_card = np.column_stack(
    [
        np.ones(len(card_data)),
        card_data["educ"].values,
        card_data["exper"].values,
        card_data["exper"].values ** 2,
    ]
)
entity_card = card_data["id"].values
card_var_names = ["const", "educ", "exper", "exper_sq"]

# Fine grid of quantiles
tau_fine = np.round(np.arange(0.05, 0.96, 0.05), 2)
print(f"Estimating QR for {len(tau_fine)} quantiles...")

# Estimate unconstrained QR
card_results = {}
for tau in tau_fine:
    model = PooledQuantile(y_card, X_card, entity_id=entity_card, quantiles=tau)
    card_results[tau] = SimpleResult(model.fit(se_type="cluster").params.ravel(), X_card)

print("Done.")

# Detect crossing
card_crossing = QuantileMonotonicity.detect_crossing(card_results, X_card)
card_crossing.summary()

In [ ]:
# Apply all three methods

# 1. Rearrangement
card_rearranged = QuantileMonotonicity.rearrangement(card_results, X_card)
card_rearr_crossing = QuantileMonotonicity.detect_crossing(card_rearranged, X_card)
print(f"After rearrangement — crossing: {card_rearr_crossing.has_crossing}")

# 2. Isotonic regression
card_coef_mat = np.array([card_results[tau].params for tau in tau_fine])
card_iso_coefs = QuantileMonotonicity.isotonic_regression(card_coef_mat, tau_fine)

card_iso_results = {}
for i, tau in enumerate(tau_fine):
    card_iso_results[tau] = SimpleResult(card_iso_coefs[i], X_card)

card_iso_crossing = QuantileMonotonicity.detect_crossing(card_iso_results, X_card)
print(f"After isotonic — crossing: {card_iso_crossing.has_crossing}")

# 3. Constrained QR (use subset for speed)
tau_coarse = np.round(np.arange(0.1, 0.91, 0.1), 2)
print(f"\nRunning constrained QR on coarser grid ({len(tau_coarse)} quantiles)...")

# Use subsample for constrained QR — full dataset is too slow
np.random.seed(42)
n_sub_card = min(100, len(y_card))
sub_idx_card = np.random.choice(len(y_card), n_sub_card, replace=False)
X_card_sub = X_card[sub_idx_card]
y_card_sub = y_card[sub_idx_card]

try:
    card_constrained = QuantileMonotonicity.constrained_qr(
        X_card_sub, y_card_sub, tau_coarse, method="SLSQP", max_iter=200
    )
    card_constr_results = {tau: SimpleResult(card_constrained[tau], X_card) for tau in tau_coarse}
    card_constr_crossing = QuantileMonotonicity.detect_crossing(card_constr_results, X_card)
    print(f"After constrained QR (subsample) — crossing: {card_constr_crossing.has_crossing}")
except Exception as e:
    print(f"Constrained QR failed: {e}")
    card_constr_results = None

In [ ]:
# Compare education coefficient paths across methods
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

educ_idx = 1  # education coefficient

# Education coefficient path
educ_unconstrained = [card_results[tau].params[educ_idx] for tau in tau_fine]
educ_rearranged = [card_rearranged[tau].params[educ_idx] for tau in tau_fine]
educ_isotonic = [card_iso_results[tau].params[educ_idx] for tau in tau_fine]

axes[0].plot(
    tau_fine,
    educ_unconstrained,
    "o-",
    linewidth=2,
    markersize=4,
    label="Unconstrained",
    color="steelblue",
)
axes[0].plot(
    tau_fine,
    educ_rearranged,
    "s-",
    linewidth=2,
    markersize=4,
    label="Rearrangement",
    color="#009E73",
)
axes[0].plot(
    tau_fine, educ_isotonic, "^-", linewidth=2, markersize=4, label="Isotonic", color="#E69F00"
)

if card_constr_results is not None:
    educ_constrained = [card_constr_results[tau].params[educ_idx] for tau in tau_coarse]
    axes[0].plot(
        tau_coarse,
        educ_constrained,
        "D-",
        linewidth=2,
        markersize=5,
        label="Constrained",
        color="#D55E00",
    )

axes[0].set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Education Coefficient", fontsize=12, fontweight="bold")
axes[0].set_title("Education Returns by Method", fontsize=14, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Experience coefficient path
exper_idx = 2
exper_unconstrained = [card_results[tau].params[exper_idx] for tau in tau_fine]
exper_rearranged = [card_rearranged[tau].params[exper_idx] for tau in tau_fine]
exper_isotonic = [card_iso_results[tau].params[exper_idx] for tau in tau_fine]

axes[1].plot(
    tau_fine,
    exper_unconstrained,
    "o-",
    linewidth=2,
    markersize=4,
    label="Unconstrained",
    color="steelblue",
)
axes[1].plot(
    tau_fine,
    exper_rearranged,
    "s-",
    linewidth=2,
    markersize=4,
    label="Rearrangement",
    color="#009E73",
)
axes[1].plot(
    tau_fine, exper_isotonic, "^-", linewidth=2, markersize=4, label="Isotonic", color="#E69F00"
)

axes[1].set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Experience Coefficient", fontsize=12, fontweight="bold")
axes[1].set_title("Experience Returns by Method", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDiscussion:")
print("  - For the Card education data, all three methods produce similar results.")
print("  - Rearrangement provides the most plausible results because it makes")
print("    the smallest adjustments while fixing monotonicity violations.")
print("  - Isotonic regression may oversmooth the coefficient paths.")
print("  - Constrained QR is the most principled but computationally expensive.")

---

## Exercise 5: Location-Scale and Crossing (Hard)

**Task**: Derive conditions under which a location-scale model never has crossing.

In [ ]:
# SOLUTION: Location-scale model and crossing
#
# MATHEMATICAL DERIVATION:
#
# The location-scale model specifies:
#   Q_tau(Y|X) = X'beta + sigma(X) * F^{-1}(tau)
#
# where sigma(X) = exp(X'gamma) > 0 is the scale function and
# F^{-1}(tau) is the quantile function of the error distribution.
#
# For tau_1 < tau_2:
#   Q_tau2(Y|X) - Q_tau1(Y|X) = sigma(X) * [F^{-1}(tau_2) - F^{-1}(tau_1)]
#
# Since F^{-1} is strictly increasing (for continuous F):
#   F^{-1}(tau_2) - F^{-1}(tau_1) > 0
#
# Therefore crossing CANNOT occur if and only if:
#   sigma(X) > 0 for all X
#
# This is guaranteed when sigma(X) = exp(X'gamma) because exp() > 0 always.
#
# KEY CONDITION: The location-scale model NEVER has crossing when the
# scale function is positive for all covariate values.
#
# If sigma(X) = X'gamma (linear, no exp), then crossing occurs when X'gamma < 0.

# NUMERICAL VERIFICATION
np.random.seed(42)
n_demo = 500
x_demo = np.random.normal(0, 1, n_demo)

# Location-scale DGP with sigma(x) = exp(0.3 + 0.5*x)
beta_loc = np.array([2.0, 1.5])  # location
gamma_scale = np.array([0.3, 0.5])  # scale
X_demo = np.column_stack([np.ones(n_demo), x_demo])

mu = X_demo @ beta_loc  # location
sigma = np.exp(X_demo @ gamma_scale)  # scale (always positive)
eps = np.random.normal(0, 1, n_demo)
y_demo = mu + sigma * eps

# True conditional quantiles (no crossing by construction)
tau_demo = np.arange(0.1, 0.91, 0.1)
true_quantiles = {}
for tau in tau_demo:
    true_quantiles[tau] = mu + sigma * stats.norm.ppf(tau)

# Verify no crossing in true model
print("True location-scale model: Checking for crossing...")
has_true_crossing = False
for i in range(len(tau_demo) - 1):
    violations = np.sum(true_quantiles[tau_demo[i]] > true_quantiles[tau_demo[i + 1]])
    if violations > 0:
        has_true_crossing = True
        print(
            f"  Crossing between tau={tau_demo[i]:.1f} and tau={tau_demo[i + 1]:.1f}: "
            f"{violations} violations"
        )

if not has_true_crossing:
    print("  No crossing in true model (as expected).")

print(f"\nsigma(X) range: [{sigma.min():.4f}, {sigma.max():.4f}]")
print(f"sigma(X) > 0 for all X: {np.all(sigma > 0)}")
print("\nConclusion: sigma(X) = exp(X'gamma) > 0 always, so crossing is impossible.")

In [ ]:
# What happens when the condition is violated?
# Use sigma(X) = X'gamma (can be negative)
gamma_bad = np.array([0.5, -0.8])  # can make sigma negative for large x
sigma_bad = X_demo @ gamma_bad  # can be negative!

# Check where sigma is negative
n_negative = np.sum(sigma_bad < 0)
print(f"Linear scale: sigma(X) negative for {n_negative}/{n_demo} observations")
print(f"sigma range: [{sigma_bad.min():.4f}, {sigma_bad.max():.4f}]")

# True quantiles with bad scale
bad_quantiles = {}
for tau in tau_demo:
    bad_quantiles[tau] = mu + sigma_bad * stats.norm.ppf(tau)

# Check crossing
n_crossing_bad = 0
for i in range(len(tau_demo) - 1):
    violations = np.sum(bad_quantiles[tau_demo[i]] > bad_quantiles[tau_demo[i + 1]] + 1e-10)
    n_crossing_bad += violations

print(f"Crossing violations with linear scale: {n_crossing_bad}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: exp(X'gamma) - no crossing
sort_idx = np.argsort(x_demo)
for tau in [0.1, 0.5, 0.9]:
    ax1.plot(x_demo[sort_idx], true_quantiles[tau][sort_idx], linewidth=2, label=f"\u03c4={tau}")
ax1.set_xlabel("x", fontsize=12)
ax1.set_ylabel("Q(\u03c4|x)", fontsize=12)
ax1.set_title("exp(X'\u03b3): No Crossing Possible", fontsize=13, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Panel B: X'gamma (linear) - crossing occurs
for tau in [0.1, 0.5, 0.9]:
    ax2.plot(x_demo[sort_idx], bad_quantiles[tau][sort_idx], linewidth=2, label=f"\u03c4={tau}")
ax2.axvline(
    x_demo[sigma_bad == sigma_bad[sigma_bad > 0].min()][0] if n_negative > 0 else 0,
    color="red",
    linestyle="--",
    alpha=0.5,
    label="\u03c3(x)=0",
)
ax2.set_xlabel("x", fontsize=12)
ax2.set_ylabel("Q(\u03c4|x)", fontsize=12)
ax2.set_title("X'\u03b3 (linear): Crossing When \u03c3 < 0", fontsize=13, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nConclusion:")
print("  The location-scale model prevents crossing when sigma(X) > 0 for all X.")
print("  Using sigma(X) = exp(X'gamma) guarantees this condition.")
print("  A linear scale sigma(X) = X'gamma can produce crossing where sigma < 0.")

---

## Exercise 6: Bootstrap with Rearrangement (Hard)

**Task**: Incorporate monotonicity correction into a bootstrap procedure and compare inference.

In [ ]:
# SOLUTION: Bootstrap with and without rearrangement
np.random.seed(42)

n_boot = 100  # Keep small for speed
tau_boot = np.round(np.arange(0.1, 0.91, 0.2), 2)  # Coarser grid for speed

# Use crossing example data
unique_entities = np.unique(entity_id)
n_entities = len(unique_entities)

# Storage for bootstrap results
boot_params_naive = {tau: [] for tau in tau_boot}
boot_params_rearranged = {tau: [] for tau in tau_boot}

print(f"Running {n_boot} bootstrap replications with {len(tau_boot)} quantiles...")
t0 = time.time()

for b in range(n_boot):
    if (b + 1) % 20 == 0:
        print(f"  Replication {b + 1}/{n_boot}")

    # Cluster bootstrap: resample entities
    boot_entities = np.random.choice(unique_entities, n_entities, replace=True)

    # Build bootstrap sample
    boot_idx = []
    for ent in boot_entities:
        boot_idx.extend(np.where(entity_id == ent)[0])
    boot_idx = np.array(boot_idx)

    X_boot = X[boot_idx]
    y_boot = y[boot_idx]

    # Estimate unconstrained QR
    boot_results = {}
    for tau in tau_boot:
        try:
            model_b = PooledQuantile(y_boot, X_boot, quantiles=tau)
            res_b = model_b.fit(se_type="nonrobust")
            params_b = res_b.params.ravel()
            boot_results[tau] = SimpleResult(params_b, X_boot)
            boot_params_naive[tau].append(params_b)
        except Exception:
            continue

    # Apply rearrangement
    if len(boot_results) == len(tau_boot):
        try:
            rearr_b = QuantileMonotonicity.rearrangement(boot_results, X_boot)
            for tau in tau_boot:
                boot_params_rearranged[tau].append(rearr_b[tau].params)
        except Exception:
            for tau in tau_boot:
                boot_params_rearranged[tau].append(boot_results[tau].params)

t_boot = time.time() - t0
print(f"\nBootstrap completed in {t_boot:.1f} seconds.")

In [ ]:
# Compare bootstrap SEs with and without rearrangement
print("BOOTSTRAP STANDARD ERRORS COMPARISON")
print("=" * 75)
print(f"{'':>6s}  {'Variable':>10s}  {'Naive SE':>12s}  {'Rearr. SE':>12s}  {'Ratio':>8s}")
print("-" * 55)

for tau in tau_boot:
    naive_arr = np.array(boot_params_naive[tau])
    rearr_arr = np.array(boot_params_rearranged[tau])

    if len(naive_arr) == 0 or len(rearr_arr) == 0:
        continue

    for v_idx, vname in enumerate(var_names):
        se_naive = np.std(naive_arr[:, v_idx], ddof=1)
        se_rearr = np.std(rearr_arr[:, v_idx], ddof=1)
        ratio = se_rearr / se_naive if se_naive > 0 else float("nan")
        print(f"{tau:6.1f}  {vname:>10s}  {se_naive:12.6f}  {se_rearr:12.6f}  {ratio:8.3f}")
    print()

In [ ]:
# Visualize bootstrap distributions for one coefficient (x1)
fig, axes = plt.subplots(1, len(tau_boot), figsize=(4 * len(tau_boot), 4))
if len(tau_boot) == 1:
    axes = [axes]

x1_idx = 1  # x1 coefficient

for ax, tau in zip(axes, tau_boot):
    naive_arr = np.array(boot_params_naive[tau])
    rearr_arr = np.array(boot_params_rearranged[tau])

    if len(naive_arr) > 0:
        ax.hist(
            naive_arr[:, x1_idx],
            bins=20,
            alpha=0.5,
            color="steelblue",
            edgecolor="black",
            label="Naive",
            density=True,
        )
    if len(rearr_arr) > 0:
        ax.hist(
            rearr_arr[:, x1_idx],
            bins=20,
            alpha=0.5,
            color="#E69F00",
            edgecolor="black",
            label="Rearranged",
            density=True,
        )

    ax.set_xlabel(f"\u03b2_x1(\u03c4={tau})", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.set_title(f"\u03c4 = {tau}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Bootstrap Distributions: x1 Coefficient", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nDiscussion:")
print("  - Rearrangement within the bootstrap typically reduces SE slightly.")
print("  - The effect is small when crossing is minor (few violations).")
print("  - With heavy crossing, rearrangement can substantially change inference.")
print("  - Recommendation: If crossing is detected, report bootstrap CIs from")
print("    the rearranged bootstrap for more reliable inference.")